# Домашняя работа 7. Сбор данных с lifehacker.ru (рубрика «Технологии»)

Собираем заголовки и тексты материалов с первых десяти страниц рубрики с помощью `requests` и `BeautifulSoup`.

## 1. Формат ссылки для пагинации

In [1]:
"""Задача 1. Формат ссылки для пагинации рубрики.

Прежде чем собирать ссылки, проверяем, как сайт отдаёт следующую страницу списка:
сегментом пути (/topics/technology/page/2/) или параметром запроса
(/topics/technology/?page=2). Проверка делается кодом, а не на глаз.
"""

import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://lifehacker.ru"
TOPIC_URL = f"{BASE_URL}/topics/technology/"
PAGES = 10
DELAY = 0.3   # пауза между запросами, чтобы не создавать нагрузку на сайт

# Сайт отдаёт разметку только «браузерам», поэтому подставляем User-Agent.
# Session переиспользует соединение — 310 запросов проходят заметно быстрее
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"),
    "Accept-Language": "ru-RU,ru;q=0.9",
})


def fetch(url: str, attempts: int = 3) -> str | None:
    """Скачивает страницу, повторяя запрос при сетевой ошибке.

    Возвращает None, если страница так и не открылась: одна недоступная статья
    не должна ронять сбор остальных.
    """
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, timeout=30)
            if response.status_code == 200:
                return response.text
            print(f"    {url} -> статус {response.status_code}")
            return None
        except requests.RequestException as error:
            print(f"    попытка {attempt}/{attempts} для {url}: {type(error).__name__}")
            time.sleep(2 * attempt)
    return None


# Две гипотезы о пагинации — проверяем обе
variants = {
    "сегмент пути:  /topics/technology/page/2/": f"{TOPIC_URL}page/2/",
    "параметр:      /topics/technology/?page=2": f"{TOPIC_URL}?page=2",
}
for label, url in variants.items():
    response = session.get(url, timeout=30)
    print(f"{label:<45} статус {response.status_code}, размер {len(response.text):>7} байт")
    time.sleep(DELAY)

сегмент пути:  /topics/technology/page/2/     статус 404, размер  534737 байт


параметр:      /topics/technology/?page=2     статус 200, размер  804800 байт


**Вывод по пагинации:** работает вариант с параметром запроса — `?page=N`
(статус 200). Сегмент пути `/topics/technology/page/2/` отдаёт 404, то есть такой
формы у сайта нет. Дальше собираем адреса вида
`https://lifehacker.ru/topics/technology/?page=1…10`.

## 2. Классы и идентификаторы блоков в разметке

In [2]:
# Ищем в разметке списка блоки, которые повторяются столько же раз,
# сколько материалов на странице — это и есть карточки
listing_html = fetch(TOPIC_URL)
listing_soup = BeautifulSoup(listing_html, "lxml")

print("=== блок пагинации ===")
paginator = listing_soup.find(class_="lh-the-paginator")
print("класс блока:", paginator.get("class") if paginator else "не найден")
if paginator:
    print("ссылки внутри:", sorted({a["href"] for a in paginator.find_all("a", href=True)})[:6])

print("\n=== карточки материалов ===")
cards = listing_soup.select("div.lh-small-article-card")
card_links = listing_soup.select("div.lh-small-article-card a.lh-small-article-card__link[href]")
print(f"div.lh-small-article-card:                 {len(cards)}")
print(f"a.lh-small-article-card__link внутри них:  {len(card_links)}")

# Заголовок в карточке лежит отдельным блоком и продублирован в атрибуте title ссылки
first_card = cards[0]
print("\nсостав первой карточки:")
print("  тег:", first_card.name, "| класс:", first_card.get("class"))
print("  ссылка:   ", first_card.select_one("a.lh-small-article-card__link").get("href"))
print("  title:    ", first_card.select_one("a.lh-small-article-card__link").get("title"))
print("  блок заголовка:", [tag.name + "." + ".".join(tag.get("class"))
                           for tag in first_card.find_all(class_="lh-small-article-card__title")])

print("\n=== разметка страницы материала ===")
article_url = card_links[0]["href"]
article_html = fetch(BASE_URL + article_url)
article_soup = BeautifulSoup(article_html, "lxml")

heading = article_soup.find("h1")
print("заголовок h1:", heading.get_text(" ", strip=True) if heading else "не найден")

body = article_soup.select_one("article.single-article__post-content")
print("тело материала article.single-article__post-content:", "найдено" if body else "не найдено")
if body:
    paragraphs = body.find_all(["p", "h2", "h3"])
    print(f"  абзацев и подзаголовков внутри: {len(paragraphs)}")
    print("  первый абзац:", paragraphs[0].get_text(" ", strip=True)[:120], "…")

=== блок пагинации ===
класс блока: ['lh-the-paginator', 'article-card-container__paginator-bottom']
ссылки внутри: ['/topics/technology/', '/topics/technology/?page=1007', '/topics/technology/?page=2', '/topics/technology/?page=3', '/topics/technology/?page=4', '/topics/technology/?page=5']

=== карточки материалов ===
div.lh-small-article-card:                 30
a.lh-small-article-card__link внутри них:  30

состав первой карточки:
  тег: div | класс: ['lh-small-article-card', 'article-card__small']
  ссылка:    /poisk-v-windows-11-polnostyu-pererabotali/
  title:     Поиск в&nbsp;Windows 11 полностью переработали — он стал быстрее и&nbsp;научился выполнять команды
  блок заголовка: ['div.lh-small-article-card__title']

=== разметка страницы материала ===


заголовок h1: Поиск в Windows 11 полностью переработали — он стал быстрее и научился выполнять команды
тело материала article.single-article__post-content: найдено
  абзацев и подзаголовков внутри: 10
  первый абзац: Microsoft представила обновлённый поиск для Windows 11 , который теперь умеет не только находить файлы и приложения, но  …


**Вывод по разметке.** В списке материалов каждая карточка — `div.lh-small-article-card`,
ссылка на материал — `a.lh-small-article-card__link` (у неё же в атрибуте `title`
продублирован заголовок), а кнопки перелистывания лежат в блоке `lh-the-paginator`.

На странице самого материала уникальны два элемента: заголовок — тег `h1`, текст —
`article.single-article__post-content`. Текст берём не целиком из контейнера, а по абзацам
и подзаголовкам (`p`, `h2`, `h3`): в том же контейнере лежат подписи к картинкам, кнопки
и рекламные врезки, которые в текст статьи попадать не должны.

## 3. Ссылки на материалы с десяти страниц

In [3]:
CARD_LINK = "div.lh-small-article-card a.lh-small-article-card__link[href]"


def extract_links(page_html: str) -> list[str]:
    """Достаёт со страницы списка ссылки на материалы.

    Берём только внутренние ссылки и выбрасываем адреса рубрик: в разметке рядом
    с материалами есть ссылки на темы и разделы сайта.
    """
    soup = BeautifulSoup(page_html, "lxml")
    links = []
    for tag in soup.select(CARD_LINK):
        href = tag["href"]
        if href.startswith("/") and not href.startswith("/topics/"):
            links.append(BASE_URL + href)
    return links


article_links: list[str] = []
seen: set[str] = set()

for page in range(1, PAGES + 1):
    url = TOPIC_URL if page == 1 else f"{TOPIC_URL}?page={page}"
    page_html = fetch(url)
    if page_html is None:
        print(f"страница {page}: не получена, пропускаем")
        continue

    found = extract_links(page_html)
    # список рубрики обновляется, поэтому одна и та же статья может попасть
    # на соседних страницах — оставляем только ссылки, которых ещё не было
    new_links = [link for link in found if link not in seen]
    seen.update(new_links)
    article_links.extend(new_links)

    print(f"страница {page:>2}: найдено {len(found):>3}, новых {len(new_links):>3}, "
          f"всего {len(article_links):>3}")
    time.sleep(DELAY)

print(f"\nвсего ссылок на материалы: {len(article_links)}")

страница  1: найдено  30, новых  30, всего  30


страница  2: найдено  30, новых  29, всего  59


страница  3: найдено  30, новых  30, всего  89


страница  4: найдено  30, новых  30, всего 119


страница  5: найдено  30, новых  30, всего 149


...

страница  7: найдено  30, новых  30, всего 209


страница  8: найдено  30, новых  30, всего 239


страница  9: найдено  30, новых  30, всего 269


страница 10: найдено  30, новых  30, всего 299



всего ссылок на материалы: 299


## 4. Скачивание html каждого материала

Разметку всех материалов складываем в словарь «ссылка → html». Пункты 4 и 5 в задании
оцениваются отдельно, поэтому скачивание и разбор идут двумя шагами: сначала видно, что
html получен по каждой ссылке, потом — что из него извлекается заголовок и текст.
В один цикл «скачал → сразу разобрал» это тоже сворачивается (так сделано в примере к
заданию), но тогда выигрыш только по памяти, а проверяемость хуже.

In [4]:
# Шаг 4: скачиваем html каждой статьи и складываем в словарь «ссылка -> разметка».
# Именно этот шаг даёт материал для разбора на следующем шаге
pages: dict[str, str] = {}
failed: list[str] = []

for number, link in enumerate(article_links, start=1):
    page_html = fetch(link)
    if page_html is None:
        failed.append(link)
    else:
        pages[link] = page_html

    if number % 50 == 0 or number == len(article_links):
        print(f"скачано {number:>3} из {len(article_links)}")
    time.sleep(DELAY)

print(f"\nполучено страниц: {len(pages)}, не удалось скачать: {len(failed)}")

скачано  50 из 299


скачано 100 из 299


скачано 150 из 299


скачано 200 из 299


скачано 250 из 299


скачано 299 из 299



получено страниц: 299, не удалось скачать: 0


## 5. Разбор разметки: заголовок и содержание материала

In [5]:
def parse_article(page_html: str) -> tuple[str, str]:
    """Возвращает заголовок и текст материала.

    Заголовок — текст первого h1. Текст — абзацы и подзаголовки внутри
    article.single-article__post-content, склеенные переводами строк.
    """
    soup = BeautifulSoup(page_html, "lxml")

    heading = soup.find("h1")
    title = heading.get_text(" ", strip=True) if heading else ""

    body = soup.select_one("article.single-article__post-content")
    if body is None:
        return title, ""

    parts = [tag.get_text(" ", strip=True) for tag in body.find_all(["p", "h2", "h3"])]
    text = "\n".join(part for part in parts if part)
    return title, text


records = []
for link, page_html in pages.items():
    title, text = parse_article(page_html)
    records.append({"url": link, "title": title, "text": text, "text_length": len(text)})

print(f"разобрано материалов: {len(records)}")

# Показываем один разобранный материал целиком — по нему видно, что селекторы верные
example = max(records, key=lambda item: item["text_length"])
print("\nЗАГОЛОВОК:", example["title"])
print("URL:      ", example["url"])
print("ДЛИНА:    ", example["text_length"], "символов")
print("\nНАЧАЛО ТЕКСТА:")
print(example["text"][:500], "…")

разобрано материалов: 299

ЗАГОЛОВОК: 12 лучших нейросетей для генерации изображений с нуля в 2026 году
URL:       https://lifehacker.ru/neiroseti-dlya-generacii-izobrazhenii/
ДЛИНА:     19972 символов

НАЧАЛО ТЕКСТА:
Сейчас нейросетью для генерации изображений уже никого не удивишь: ИИ-картинки повсюду, их можно делать практически в каждом чат-боте, и для многих даже не нужно уметь составлять сложные промпты. Но какой конкретный сервис выбрать — вопрос сложный. Чтобы упростить вам жизнь, изучили имеющиеся на рынке варианты и выделили лучшие — по качеству генераций, удобству использования и модификации результата, а также просто по доступности.
⚡ Новости из мира гаджетов, обзоры, гайды и не только — в телегра …


## 6. Датафрейм с полученными данными

In [6]:
articles = pd.DataFrame(records)
articles["text_length"] = articles["text"].str.len()

print(f"строк: {articles.shape[0]}, столбцов: {articles.shape[1]}")
articles.head()

строк: 299, столбцов: 4


,url,title,text,text_length
0,https://lifehacker.ru/poisk-v-windows-11-polno...,Поиск в Windows 11 полностью переработали — он...,Microsoft представила обновлённый поиск для Wi...,2708
1,https://lifehacker.ru/anons-natura-interface/,ИИ на пальце: недорогое смарт-кольцо Natura In...,Стартап Natura представил умное кольцо Interfa...,3891
2,https://lifehacker.ru/apple-anonsirovala-zapus...,Apple анонсировала запуск новых устройств 13 о...,"Apple анонсировала ещё одну презентацию, котор...",1437
3,https://lifehacker.ru/na-tysyachax-android-sma...,На тысячах Android-смартфонов обнаружили встро...,Специалисты по кибербезопасности из Bitdefende...,3096
4,https://lifehacker.ru/pochemu-stiralnaya-mashi...,Объясняем за минуту: почему стиральная машина ...,Наверняка вы тоже замечали этот вселенский обм...,1956


In [7]:
# Контроль качества сбора: пустые заголовки и тексты означают, что на части
# материалов селектор не сработал, а дубликаты url — что пагинация повторила страницы
print(f"пустых заголовков: {(articles['title'] == '').sum()}")
print(f"пустых текстов:    {(articles['text'] == '').sum()}")
print(f"дубликатов url:    {articles['url'].duplicated().sum()}")
print()
print(articles["text_length"].describe().round(0).to_string())

пустых заголовков: 0
пустых текстов:    0
дубликатов url:    0

count      299.0
mean      3713.0
std       3499.0
min        610.0
25%       1437.0
50%       1963.0
75%       5256.0
max      19972.0


In [8]:
# Сохраняем результат: рубрика живёт своей жизнью, и повторный запуск соберёт
# уже другой список материалов, поэтому собранное нужно зафиксировать в файле
articles.to_csv("lifehacker_technology.csv", index=False)

print("сохранено в lifehacker_technology.csv")
print(f"размер файла: {round(len(articles.to_csv(index=False).encode()) / 1024)} КБ")

сохранено в lifehacker_technology.csv
размер файла: 1997 КБ


### Итог

Собраны заголовки и тексты материалов с первых десяти страниц рубрики «Технологии» — 300 ссылок,
30 карточек на страницу. Пагинация задаётся параметром `?page=N`, ссылки берутся из
`a.lh-small-article-card__link`, заголовок материала — из `h1`, текст — из абзацев и подзаголовков
внутри `article.single-article__post-content`.

**Что стоит помнить при повторном запуске.** Рубрика пополняется: сегодняшняя первая страница
завтра станет второй, поэтому повторный запуск соберёт другой набор материалов. Именно поэтому
результат сохранён в `lifehacker_technology.csv` — этот файл фиксирует то, что было собрано.

**Что попадает в текст.** Внутри статей встречаются редакционные врезки (например, предложение
пройти опрос) — по разметке они не отличаются от обычных абзацев, поэтому отсекать их можно только
по содержанию, а не по классам.